In [18]:
import boto3
import sagemaker
import json
import time
from sagemaker.processing import ScriptProcessor

In [2]:
session = boto3.Session()
account_id = session.client('sts').get_caller_identity().get('Account')
region = session.region_name

# SageMaker session
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()

print(f"Account ID: {account_id}")
print(f"Region: {region}")
print(f"Role: {role}")

Account ID: 755352605209
Region: ca-central-1
Role: arn:aws:iam::755352605209:role/sagemaker-training-role


In [3]:
# Get the list of ingested dates
s3_client = boto3.client('s3')
bucket_raw = "forest-carbon-dung-raw-extended"

def get_ingested_dates(region_name):
    prefix = f"raw/region={region_name}/date="
    dates = set()
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket_raw, Prefix=prefix):
        if 'Contents' not in page:
            continue
        for obj in page['Contents']:
            parts = obj['Key'].split('/')
            for part in parts:
                if part.startswith('date='):
                    date_str = part.split('=')[1]
                    dates.add(date_str)
                    break
    return sorted(list(dates))

# Lấy dates cho từng region
amazon_dates = get_ingested_dates('amazon')
vietnam_dates = get_ingested_dates('vietnam')
africa_dates = get_ingested_dates('central_africa')

print(f"Amazon: {len(amazon_dates)} images")
print(f"Vietnam: {len(vietnam_dates)} images")
print(f"Central Africa: {len(africa_dates)} images")
print(f"\nFirst 5 Amazon dates: {amazon_dates[:5]}")

Amazon: 160 images
Vietnam: 143 images
Central Africa: 90 images

First 5 Amazon dates: ['2018-01-06', '2018-02-07', '2018-03-02', '2018-05-11', '2018-06-02']


In [11]:
# ECR address
ecr_image = f"{account_id}.dkr.ecr.{region}.amazonaws.com/sagemaker-chip-processor:latest"
print(f"ECR image: {ecr_image}")

# create processor
script_processor = ScriptProcessor(
    image_uri=ecr_image,
    command=['python3'],  # command hoạt động trong ScriptProcessor
    role=role,
    instance_count=1,
    instance_type='ml.r5.xlarge',
    volume_size_in_gb=250,
    max_runtime_in_seconds=86400,
    sagemaker_session=sagemaker_session
)
print(f"Processor created succesfully")

ECR image: 755352605209.dkr.ecr.ca-central-1.amazonaws.com/sagemaker-chip-processor:latest
Processor created succesfully


In [15]:
# Save dates for each region
import json
import boto3
s3_client= boto3.client("s3")
bucket_for_config = "forest-carbon-dung-raw-extended"

# Save the dates!
for region_name, dates in [("amazon", amazon_dates), ("vietnam", vietnam_dates), ("central_africa", africa_dates)]:
    config_key = f"config/{region_name}_dates.json"
    s3_client.put_object(
        Bucket=bucket_for_config,
        Key=config_key,
        Body=json.dumps(dates)
    )
    print(f"✅ Uploaded {len(dates)} dates to s3://{bucket_for_config}/{config_key}")

✅ Uploaded 160 dates to s3://forest-carbon-dung-raw-extended/config/amazon_dates.json
✅ Uploaded 143 dates to s3://forest-carbon-dung-raw-extended/config/vietnam_dates.json
✅ Uploaded 90 dates to s3://forest-carbon-dung-raw-extended/config/central_africa_dates.json


In [ ]:
# Run job for Amazon
print("🚀 Starting Processing Job for Amazon")
print(f"Processing {len(amazon_dates)} images")
config_bucket = "forest-carbon-dung-raw-extended"

# Hàm chạy job cho 1 region
def run_processing_job(region_name, dates):
    print(f"\n{'='*50}")
    print(f"Processing {region_name} with {len(dates)} images")
    print(f"{'='*50}")
    
    script_processor.run(
        code='processing_script.py',
        arguments=[
            '--region', region_name,
            '--config-bucket', config_bucket,
            '--raw-bucket', 'forest-carbon-dung-raw-extended',
            '--proc-bucket', 'forest-carbon-dung-processed-extended'
        ],
        wait=True,  # Wait for job to complete
        logs=True
    )
    print(f"✅ {region_name} completed!")
    time.sleep(10)  # Rest for 10 sec between jobs

# Chạy lần lượt (nếu muốn)
run_processing_job('amazon', amazon_dates)
run_processing_job('vietnam', vietnam_dates)
run_processing_job('central_africa', africa_dates)

INFO:sagemaker:Creating processing-job with name sagemaker-chip-processor-2026-04-26-23-18-45-679


🚀 Starting Processing Job for Amazon
Processing 160 images

Processing amazon with 160 images
......./usr/local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
Processing 160 images for amazon
  Processing amazon 2018-01-06
  ✅ amazon 2018-01-06: 1742 chips, 22 skipped
  Processing amazon 2018-02-07
  ✅ amazon 2018-02-07: 1742 chips, 22 skipped
  Processing amazon 2018-03-02
  ✅ amazon 2018-03-02: 1742 chips, 22 skipped
  Processing amazon 2018-05-11
  ✅ amazon 2018-05-11: 1742 chips, 22 skipped
  Processing amazon 2018-06-02
  ✅ amazon 2018-06-02: 1742 chips, 22 skipped
  Processing amazon 2018-06-10
  ✅ amazo

INFO:sagemaker:Creating processing-job with name sagemaker-chip-processor-2026-04-27-06-49-26-604



Processing vietnam with 143 images
......./usr/local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
Processing 143 images for vietnam
  Processing vietnam 2018-01-03
  ✅ vietnam 2018-01-03: 1759 chips, 5 skipped
  Processing vietnam 2018-01-08
  ✅ vietnam 2018-01-08: 1759 chips, 5 skipped
  Processing vietnam 2018-02-02
  ✅ vietnam 2018-02-02: 1759 chips, 5 skipped
  Processing vietnam 2018-02-07
  ✅ vietnam 2018-02-07: 1759 chips, 5 skipped
  Processing vietnam 2018-02-12
  ✅ vietnam 2018-02-12: 1759 chips, 5 skipped
  Processing vietnam 2018-02-17
  ✅ vietnam 2018-02-17: 1759 chips, 5 skipped
  Processing v

In [20]:
import boto3

s3 = boto3.client("s3")
bucket = "forest-carbon-dung-processed-extended"
print(f"🔍 Counting the number of chips presenting")

image_files = []
mask_files = []
region_stats = {}

paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=bucket, Prefix="processed/"):
    if 'Contents' not in page:
        continue
    for obj in page['Contents']:
        key = obj['Key']
        if 'images/chip_' in key:
            image_files.append(key)
            # Extract region
            region = key.split('/')[1].split('=')[1]
            if region not in region_stats:
                region_stats[region] = 0
            region_stats[region] +=1
        elif 'masks/mask_' in key:
            mask_files.append(key)

print(f"📊Stats Results:")
print(f" Total number of chips:  {len(image_files):,}")
print(f" Total number of mask chips:  {len(mask_files):,}")
print(f" Total number of (image+ mask):  {len(mask_files):,}")

print(f"\n Details regarding Regions:")
for region, count in region_stats.items():
    print(f"{region.upper()}: {count:,} chips")

print(f"\n📁 MẪU DỮ LIỆU (3 chips đầu tiên):")
for img in image_files[:3]:
    print(f"  {img}")   

🔍 Counting the number of chips presenting
📊Stats Results:
 Total number of chips:  650,356
 Total number of mask chips:  650,356
 Total number of (image+ mask):  650,356

 Details regarding Regions:
AMAZON: 276,978 chips
CENTRAL_AFRICA: 121,841 chips
VIETNAM: 251,537 chips

📁 MẪU DỮ LIỆU (3 chips đầu tiên):
  processed/region=amazon/date=2018-01-06/images/chip_0.npy
  processed/region=amazon/date=2018-01-06/images/chip_1.npy
  processed/region=amazon/date=2018-01-06/images/chip_10.npy
